# Move Abs Notebook

Absolute one-axis moves through `kohdalab.api.Experiment`.

Edit `config_path`, `coordinate`, and `targets` before running on hardware. This notebook uses `auto_connect=False`; review the config, then connect devices explicitly before running a move.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from kohdalab.api import Experiment, format_move_abs_row, load_config, move_abs_row_from_position


config_path: Path | None = None  # Set a lab-specific JSON path before hardware use.
config = load_config() if config_path is None else load_config(config_path)
experiment = Experiment(config, auto_connect=False)
# Review config, then run experiment.connect_all() before moving hardware.
zero = config["measurements"]["move_abs"]["zero"]
configured_targets = config["measurements"]["move_abs"].get("targets", {})


In [ ]:
coordinate = "measurement"
targets = {
    "t": float(configured_targets.get("t", zero.get("t_ps", 0.0))),
    "x": float(configured_targets.get("x", zero.get("x_um", 0.0))),
    "y": float(configured_targets.get("y", zero.get("y_um", 0.0))),
}
axes_to_move = ["t", "x", "y"]


In [ ]:
rows = []
for axis in axes_to_move:
    target = targets[axis]
    if axis == "t":
        position = experiment.move_delay_stage(target, coordinate=coordinate)
    else:
        position = experiment.move_scanner(axis, target, coordinate=coordinate)
    row = move_abs_row_from_position(axis, position, target=target, coordinate=coordinate, zero=zero)
    rows.append(row)
    print(format_move_abs_row(row, index=len(rows), total=len(axes_to_move)), flush=True)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
